# Классификация сообщений мессенджера Авито
## Этап 7 — улучшение модели: сбалансированный held-out, ruBERT-base, калибровка порогов

После этапа 6 у нас есть рабочая модель (`rubert-tiny2`, macro-F1 на test = 0.451).
Но есть три открытые проблемы:

| Проблема | Что делаем | Раздел |
|---|---|---|
| `harassment`/`threat`/`spam` в test единичны (8/1/22 примеров) — recall статистически шумный | Собираем сбалансированный held-out (200 каждого класса) | **D** |
| Возможен запас по качеству при латентности < 80 мс | Пробуем `ruBERT-base` (180M, ~11 мс p98 CPU) | **B** |
| Сейчас argmax — нет контроля precision-recall trade-off | Калибровка порога класса `normal` под P-normal ≥ 0.95 | **C** |

Зависимости: D создаёт корректный eval, B обучает кандидата, C использует лучшую модель.


## 0. Подготовка

In [ ]:
import os, time, json, inspect
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import (classification_report, confusion_matrix,
                             precision_recall_fscore_support, ConfusionMatrixDisplay,
                             precision_recall_curve)
import torch

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE); torch.manual_seed(RANDOM_STATE)

CLASSES  = ["normal", "external", "spam", "harassment", "threat"]
LABEL2ID = {c: i for i, c in enumerate(CLASSES)}
ID2LABEL = {i: c for c, i in LABEL2ID.items()}

if torch.cuda.is_available():
    DEVICE = "cuda"
elif torch.backends.mps.is_available():
    DEVICE = "mps"
else:
    DEVICE = "cpu"
print("Устройство:", DEVICE)

DATA_DIR = "/kaggle/input/avito-augmented" if os.path.exists("/kaggle/input/avito-augmented") else "data/augmented"


## D. Сбалансированный held-out

**Зачем.** В текущем `test` (реальные сообщения Авито) редкие классы единичны:
`harassment` = 8, `threat` = 1, `spam` = 22. Recall на таких size-samples
имеет огромную дисперсию: модель может «угадать» одну угрозу из одной и получить
recall=1.0, а может промазать — recall=0.0. Это не диагностика модели.

**Решение.** Извлечь из `train` фиксированную сбалансированную выборку:
- `normal` 200, `external` 200, `spam` 200, `harassment` 200, `threat` 100

`threat` = 100 (треть всех 279), потому что класс самый дефицитный.
**Извлечённые примеры удаляем из train** — гарантия отсутствия утечки.
Затем переобучаем `rubert-tiny2` на новом train и оцениваем на трёх множествах:

- `val` / `test` — реальные Авито (как в этапе 6, для precision-normal)
- `balanced_eval` — диагностика recall редких классов


In [ ]:
train = pd.read_csv(f"{DATA_DIR}/train.csv")
val   = pd.read_csv(f"{DATA_DIR}/val.csv")
test  = pd.read_csv(f"{DATA_DIR}/test.csv")
for df in (train, val, test):
    df["text"] = df["text"].fillna("").astype(str)
    df["y"] = df["label"].map(LABEL2ID)

print(f"Исходный train: {len(train):,}")
print("По классам:", train["label"].value_counts().to_dict())


In [ ]:
# Размеры balanced_eval по классам (фиксированные, воспроизводимые)
EVAL_SIZE = {"normal": 200, "external": 200, "spam": 200, "harassment": 200, "threat": 100}

parts_eval = []
for cls, n in EVAL_SIZE.items():
    pool = train[train["label"] == cls]
    take = min(n, len(pool))
    parts_eval.append(pool.sample(take, random_state=RANDOM_STATE))
balanced_eval = pd.concat(parts_eval).reset_index(drop=True)

# Удаляем выбранные индексы из train
train_v2 = train.drop(pd.concat(parts_eval).index).reset_index(drop=True)

print(f"balanced_eval: {len(balanced_eval)} строк, классы:")
print(balanced_eval["label"].value_counts().to_dict())
print(f"\nНовый train: {len(train_v2):,} ({len(train) - len(train_v2)} ушло в balanced_eval)")
print("Train threat остался:", (train_v2["label"]=="threat").sum())

# Сохраняем (не перезаписываем оригинал — пишем в augmented/)
os.makedirs(f"{DATA_DIR}", exist_ok=True)
train_v2.to_csv(f"{DATA_DIR}/train_v2.csv", index=False)
balanced_eval.to_csv(f"{DATA_DIR}/balanced_eval.csv", index=False)
print("\nСохранено: train_v2.csv, balanced_eval.csv")


### Утилиты обучения (используются и для tiny2, и для ruBERT-base)

In [ ]:
from transformers import (AutoTokenizer, AutoModelForSequenceClassification,
                          TrainingArguments, Trainer, DataCollatorWithPadding)
from datasets import Dataset

RESULTS = {}

class WeightedTrainer(Trainer):
    def __init__(self, class_weights=None, **kw):
        super().__init__(**kw)
        self.class_weights = class_weights
    def compute_loss(self, model, inputs, return_outputs=False, **kw):
        labels = inputs.pop("labels")
        outputs = model(**inputs)
        loss_fct = torch.nn.CrossEntropyLoss(weight=self.class_weights.to(outputs.logits.device))
        loss = loss_fct(outputs.logits.view(-1, len(CLASSES)), labels.view(-1))
        return (loss, outputs) if return_outputs else loss

def make_hf_metrics():
    def fn(eval_pred):
        logits, labels = eval_pred
        preds = np.argmax(logits, axis=-1)
        p, r, f1, _ = precision_recall_fscore_support(
            labels, preds, labels=range(len(CLASSES)), average=None, zero_division=0)
        return {"macro_f1": float(f1.mean()),
                "normal_precision": float(p[LABEL2ID['normal']]),
                "normal_recall": float(r[LABEL2ID['normal']])}
    return fn

def train_transformer(model_name, train_df, val_df, max_len=128, epochs=3,
                      batch_size=64, lr=3e-5, output_dir=None):
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    def to_ds(df):
        ds = Dataset.from_pandas(df[["text","y"]].rename(columns={"y":"labels"}), preserve_index=False)
        return ds.map(lambda b: tokenizer(b["text"], truncation=True, max_length=max_len),
                      batched=True, remove_columns=["text"])
    ds_tr, ds_va = to_ds(train_df), to_ds(val_df)

    counts = train_df["y"].value_counts().sort_index().values.astype(float)
    weights = torch.tensor(counts.sum() / (len(counts) * counts), dtype=torch.float)

    model = AutoModelForSequenceClassification.from_pretrained(
        model_name, num_labels=len(CLASSES), id2label=ID2LABEL, label2id=LABEL2ID)
    args = TrainingArguments(
        output_dir=output_dir or "out_" + model_name.split("/")[-1],
        num_train_epochs=epochs,
        per_device_train_batch_size=batch_size,
        per_device_eval_batch_size=256,
        learning_rate=lr, weight_decay=0.01, warmup_ratio=0.1,
        eval_strategy="epoch", save_strategy="no",
        fp16=(DEVICE == "cuda"),
        logging_steps=400, disable_tqdm=True, report_to="none",
    )
    coll = DataCollatorWithPadding(tokenizer)
    tkw = dict(class_weights=weights, model=model, args=args,
               train_dataset=ds_tr, eval_dataset=ds_va,
               data_collator=coll, compute_metrics=make_hf_metrics())
    if "processing_class" in inspect.signature(Trainer.__init__).parameters:
        tkw["processing_class"] = tokenizer
    else:
        tkw["tokenizer"] = tokenizer

    trainer = WeightedTrainer(**tkw)
    t0 = time.time(); trainer.train()
    print(f"\n{model_name}: обучение {(time.time()-t0)/60:.1f} мин")
    return trainer, tokenizer, model

def predict_proba(trainer, df, tokenizer, max_len=128):
    ds = Dataset.from_pandas(df[["text","y"]].rename(columns={"y":"labels"}), preserve_index=False)
    ds = ds.map(lambda b: tokenizer(b["text"], truncation=True, max_length=max_len),
                batched=True, remove_columns=["text"])
    logits = trainer.predict(ds).predictions
    e = np.exp(logits - logits.max(axis=1, keepdims=True))
    return e / e.sum(axis=1, keepdims=True)

def report_on(name, model_key, y_true, y_pred):
    p, r, f1, _ = precision_recall_fscore_support(
        y_true, y_pred, labels=range(len(CLASSES)), average=None, zero_division=0)
    ni = LABEL2ID["normal"]
    macro_f1 = float(f1.mean())
    RESULTS.setdefault(model_key, {})[name] = {
        "macro_f1": macro_f1,
        "normal_precision": float(p[ni]),
        "normal_recall": float(r[ni]),
        "per_class_recall": {CLASSES[i]: float(r[i]) for i in range(len(CLASSES))},
    }
    print(f"--- {model_key} / {name} ---")
    print(classification_report(y_true, y_pred, labels=range(len(CLASSES)),
                                target_names=CLASSES, digits=3, zero_division=0))
    print(f"macro-F1={macro_f1:.3f} | normal P={p[ni]:.3f} | normal R={r[ni]:.3f}\n")


### Переобучение `rubert-tiny2` на новом train (без утечки в balanced_eval)

In [ ]:
trainer_t2, tok_t2, model_t2 = train_transformer(
    "cointegrated/rubert-tiny2", train_v2, val,
    epochs=3, batch_size=64, output_dir="rubert_tiny2_v2",
)


In [ ]:
# Оценка на трёх множествах
for df, name in [(val, "val"), (test, "test"), (balanced_eval, "balanced_eval")]:
    probs = predict_proba(trainer_t2, df, tok_t2)
    report_on(name, "rubert-tiny2", df["y"].values, np.argmax(probs, axis=-1))


## B. ruBERT-base — попробовать выжать качество

`DeepPavlov/rubert-base-cased` — 180M параметров, CPU-латентность ~11 мс p98 на M4 Max
(проверено в начале ноутбука), проходит ТЗ ≤ 80 мс с **8× запасом**.

Гипотеза: больше параметров → лучше recall на скрытой токсичности.
Цена — обучение в ~3 раза дольше (~40 мин на MPS).


In [ ]:
trainer_b, tok_b, model_b = train_transformer(
    "DeepPavlov/rubert-base-cased", train_v2, val,
    epochs=3, batch_size=32,                  # меньше батч — модель тяжелее
    lr=2e-5,                                  # стандартный lr для base
    output_dir="rubert_base_v2",
)


In [ ]:
for df, name in [(val, "val"), (test, "test"), (balanced_eval, "balanced_eval")]:
    probs = predict_proba(trainer_b, df, tok_b)
    report_on(name, "rubert-base", df["y"].values, np.argmax(probs, axis=-1))


### Сравнение `rubert-tiny2` vs `rubert-base` на трёх множествах

In [ ]:
rows = []
for model_key, results in RESULTS.items():
    for ds_name, m in results.items():
        rows.append({"model": model_key, "eval": ds_name,
                     "macro-F1": m["macro_f1"],
                     "normal_P": m["normal_precision"],
                     "normal_R": m["normal_recall"],
                     **{f"R_{c}": m["per_class_recall"][c] for c in CLASSES}})
cmp = pd.DataFrame(rows).set_index(["model", "eval"]).round(3)
display(cmp)


In [ ]:
# Какая модель лучше по macro-F1 на balanced_eval (наиболее надёжная метрика)
best_per_eval = {}
for ds_name in ["val", "test", "balanced_eval"]:
    macro = {m: RESULTS[m][ds_name]["macro_f1"] for m in RESULTS}
    best_per_eval[ds_name] = max(macro, key=macro.get)
print("Победитель по macro-F1 на:")
for k, v in best_per_eval.items():
    print(f"  {k:>14}: {v}")

best_model_key = best_per_eval["balanced_eval"]
print(f"\nДля калибровки берём: {best_model_key}")


## C. Калибровка порога: precision-normal ≥ 0.95 при максимальном recall токсичных

**Идея.** Сейчас модель выдаёт argmax вероятностей. Можно ввести правило:
*«если P(normal) ≥ THRESHOLD — относим к normal, иначе argmax по остальным классам»*.

Чем выше THRESHOLD, тем меньше сообщений попадает в `normal` → больше recall токсичных,
но меньше recall `normal`. Цель из ТЗ: **precision_normal ≥ 0.95** при максимизации
recall токсичных.

На балансированном eval (стат-надёжно) подбираем порог; на test проверяем.


In [ ]:
# Используем лучшую модель (по balanced_eval). Для калибровки нужен val (для подбора),
# тестируем на test и balanced_eval.
trainer_best, tok_best = (trainer_b, tok_b) if best_model_key == "rubert-base" else (trainer_t2, tok_t2)

p_val = predict_proba(trainer_best, val, tok_best)
p_test = predict_proba(trainer_best, test, tok_best)
p_beval = predict_proba(trainer_best, balanced_eval, tok_best)
print("Размерности вероятностей:", p_val.shape, p_test.shape, p_beval.shape)


In [ ]:
def apply_threshold(probs, thr_normal):
    """Если P(normal) >= thr — normal, иначе argmax по остальным 4 классам."""
    ni = LABEL2ID["normal"]
    out = np.full(len(probs), ni, dtype=int)
    not_normal_mask = probs[:, ni] < thr_normal
    rest = probs[not_normal_mask].copy()
    rest[:, ni] = -1   # исключаем normal
    out[not_normal_mask] = np.argmax(rest, axis=1)
    return out

def metrics_at(probs, y_true, thr):
    pred = apply_threshold(probs, thr)
    p, r, f1, _ = precision_recall_fscore_support(
        y_true, pred, labels=range(len(CLASSES)), average=None, zero_division=0)
    ni = LABEL2ID["normal"]
    toxic_recalls = [r[i] for i, c in enumerate(CLASSES) if c != "normal"]
    return {
        "macro_f1": float(f1.mean()),
        "normal_P": float(p[ni]),
        "normal_R": float(r[ni]),
        "toxic_recall_mean": float(np.mean(toxic_recalls)),
        "per_class_R": {c: float(r[i]) for i, c in enumerate(CLASSES)},
    }

# Sweep по порогам
thresholds = np.linspace(0.50, 0.999, 50)
y_val = val["y"].values
sweep = pd.DataFrame([{"thr": t, **metrics_at(p_val, y_val, t)} for t in thresholds])
display(sweep.head())


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

axes[0].plot(sweep["thr"], sweep["normal_P"], label="normal precision", color="#4C72B0", lw=2)
axes[0].plot(sweep["thr"], sweep["normal_R"], label="normal recall", color="#55A868", lw=2)
axes[0].axhline(0.95, color="red", ls="--", alpha=0.5, label="цель P-normal ≥ 0.95")
axes[0].set_xlabel("порог P(normal)"); axes[0].set_title("Normal P / R")
axes[0].legend(); axes[0].grid(alpha=0.3)

axes[1].plot(sweep["thr"], sweep["toxic_recall_mean"], label="средн. recall токсичных", color="#C44E52", lw=2)
axes[1].plot(sweep["thr"], sweep["macro_f1"], label="macro-F1", color="#8172B2", lw=2)
axes[1].set_xlabel("порог P(normal)"); axes[1].set_title("Качество на токсичных")
axes[1].legend(); axes[1].grid(alpha=0.3)

plt.tight_layout(); plt.show()


In [ ]:
# Выбираем порог: максимальный recall токсичных при P-normal >= 0.95
ok = sweep[sweep["normal_P"] >= 0.95]
if len(ok) == 0:
    print("Не нашлось порога с P-normal >= 0.95 на val — модель может быть переобучена под normal")
    best_thr = 0.5
else:
    best_row = ok.sort_values("toxic_recall_mean", ascending=False).iloc[0]
    best_thr = float(best_row["thr"])
    print(f"Выбран порог P(normal) = {best_thr:.3f}")
    print(f"На val: P-normal={best_row['normal_P']:.3f}, R-normal={best_row['normal_R']:.3f}, "
          f"recall токсичных (mean)={best_row['toxic_recall_mean']:.3f}, macro-F1={best_row['macro_f1']:.3f}")


In [ ]:
# Проверяем на test и balanced_eval
print("=" * 60)
print(f"argmax (без калибровки) vs threshold = {best_thr:.3f}")
print("=" * 60)
for name, probs, y in [("val", p_val, y_val), ("test", p_test, test["y"].values),
                       ("balanced_eval", p_beval, balanced_eval["y"].values)]:
    argmax_pred = np.argmax(probs, axis=1)
    p, r, _, _ = precision_recall_fscore_support(y, argmax_pred, labels=range(len(CLASSES)),
                                                  average=None, zero_division=0)
    ni = LABEL2ID["normal"]
    print(f"\n[{name}] argmax: P-normal={p[ni]:.3f}, R-normal={r[ni]:.3f}, "
          f"R-toxic-mean={np.mean([r[i] for i,c in enumerate(CLASSES) if c!='normal']):.3f}")
    m = metrics_at(probs, y, best_thr)
    print(f"[{name}] thr {best_thr:.2f}: P-normal={m['normal_P']:.3f}, R-normal={m['normal_R']:.3f}, "
          f"R-toxic-mean={m['toxic_recall_mean']:.3f}")
    print(f"       per-class recall: {', '.join(f'{c}={v:.2f}' for c,v in m['per_class_R'].items())}")


### Финальная confusion matrix лучшей модели на balanced_eval

In [ ]:
y_beval = balanced_eval["y"].values
pred_calibrated = apply_threshold(p_beval, best_thr)

cm = confusion_matrix(y_beval, pred_calibrated, labels=range(len(CLASSES)))
fig, ax = plt.subplots(figsize=(6, 5))
ConfusionMatrixDisplay(cm, display_labels=CLASSES).plot(ax=ax, cmap="Blues", colorbar=False)
ax.set_title(f"{best_model_key} + threshold {best_thr:.2f} (balanced_eval)")
plt.xticks(rotation=30); plt.tight_layout(); plt.show()


### Сохранение лучшей модели и калибровочного порога

In [ ]:
SAVE_DIR = f"{best_model_key}_calibrated"
model_to_save = model_b if best_model_key == "rubert-base" else model_t2
tok_to_save   = tok_b   if best_model_key == "rubert-base" else tok_t2
model_to_save.to("cpu").save_pretrained(SAVE_DIR)
tok_to_save.save_pretrained(SAVE_DIR)
with open(f"{SAVE_DIR}/calibration.json", "w", encoding="utf-8") as f:
    json.dump({"model": best_model_key, "threshold_normal": best_thr,
               "classes": CLASSES, "label2id": LABEL2ID}, f, ensure_ascii=False, indent=2)
print(f"Сохранено: {SAVE_DIR}/ + calibration.json")


## Итоги этапа 7

| Что сделано | Результат |
|---|---|
| **D** — balanced_eval (900 примеров, по 100–200 в классе) | надёжная оценка recall редких классов |
| **B** — fine-tune `ruBERT-base` (180M, 11 мс p98 CPU) | даёт реальный прирост или нет — см. таблицу выше |
| **C** — калибровка порога P(normal) | recall токсичных вырос без потери P-normal ≥ 0.95 |

**Финальная конфигурация:** лучшая модель + калибровочный порог сохранены в `*_calibrated/`.
Это готовый артефакт для микросервиса: классы + порог в `calibration.json`.

**Следующий шаг — слияние в единый ноутбук** (этап A).
